# Finetuning Parakeet on ATC radio comms 🛩️

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xnevar00/ASR-finetune-demo/blob/main/finetune_demo.ipynb)

Today we'll take NVIDIA's **Parakeet** ASR model — trained on generic English speech — and finetune it on a tiny set of real Czech air-traffic-control radio recordings (Kunovice airfield). The vocabulary here is *nothing* like what Parakeet has seen: NATO-alphabet callsigns, runway numbers spoken as digits, aviation jargon (`QNH`, `CAVOK`, `CTR`...), all in Czech.

So this won't produce a production-grade model from 12 clips — it *will* clearly show the model shift toward the new vocabulary, which is the point.

**Plan:**
1. Baseline: see Parakeet fail on our data as-is
2. Look at the data through **Lhotse** (play clips, inspect cuts, duration stats)
3. Export to a training manifest, watch training live in **Weights & Biases**
4. Finetune (a few minutes, small subset)
5. Same clip, before vs. after — side by side

⚠️ Before the workshop: run this notebook top-to-bottom once yourself on a GPU runtime (`Runtime → Change runtime type → GPU`) to make sure install/versions still match — NeMo's API shifts between releases.

## 0. Setup

In [ ]:
# Takes a few minutes on Colab — a good moment for a coffee / intro slides
!pip install -q "nemo_toolkit[asr]" lhotse wandb pandas matplotlib

In [ ]:
# Pull the workshop repo (notebook + audio/ + transcripts.csv) from GitHub
REPO_URL = "https://github.com/xnevar00/ASR-finetune-demo.git"
!git clone -q $REPO_URL repo
%cd repo

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import Audio, HTML, display

AUDIO_DIR = Path("audio")
df = pd.read_csv(AUDIO_DIR / "transcripts.csv")
df

## 1. Baseline — how bad is "out of the box" Parakeet on this?

We hold one clip out for evaluation (the longest, most sentence-like one) and never train on it. Everything else becomes our (tiny!) training set.

In [ ]:
EVAL_FILENAME = "23-07-21_11-05-03-0_001.wav"  # the VFR-flight-plan clip: longest, richest sentence

eval_row = df[df.filename == EVAL_FILENAME].iloc[0]
train_df = df[df.filename != EVAL_FILENAME].reset_index(drop=True)

print("Eval clip:", eval_row.filename)
print("Reference:", eval_row.text)
display(Audio(str(AUDIO_DIR / eval_row.filename)))

In [ ]:
import nemo.collections.asr as nemo_asr

# Swap for a smaller checkpoint (e.g. "nvidia/parakeet-tdt-0.6b-v2") if download/inference is too slow on the day
MODEL_NAME = "nvidia/parakeet-tdt-1.1b"
asr_model = nemo_asr.models.ASRModel.from_pretrained(MODEL_NAME)

In [ ]:
baseline_text = asr_model.transcribe([str(AUDIO_DIR / eval_row.filename)])[0].text
print("Reference:", eval_row.text)
print("Baseline :", baseline_text)

## 2. Look at the data through Lhotse

Instead of just pointing at file paths, let's build a proper `CutSet` — Lhotse's core abstraction that pairs audio with its supervision (transcript, speaker, language...). This is also the object NeMo's newer dataloaders consume directly.

In [ ]:
from lhotse import Recording, RecordingSet, SupervisionSegment, SupervisionSet, CutSet

recordings = RecordingSet.from_recordings(
    Recording.from_file(AUDIO_DIR / row.filename, recording_id=row.filename)
    for row in df.itertuples()
)
supervisions = SupervisionSet.from_segments(
    SupervisionSegment(
        id=row.filename,
        recording_id=row.filename,
        start=0.0,
        duration=recordings[row.filename].duration,
        text=row.text,
        language="cs",
    )
    for row in df.itertuples()
)
cuts = CutSet.from_manifests(recordings=recordings, supervisions=supervisions)
cuts.describe()

In [ ]:
# One cut, up close
example_cut = [c for c in cuts if c.recording_id == EVAL_FILENAME][0]
print(example_cut)
example_cut.plot_audio()

In [ ]:
# A few random clips, audio + text side by side
import random

for cut in random.sample(list(cuts), 3):
    print(f"{cut.id}  ({cut.duration:.1f}s)")
    print("  \u2192", cut.supervisions[0].text)
    display(Audio(cut.load_audio().squeeze(), rate=cut.sampling_rate))

In [ ]:
import matplotlib.pyplot as plt

durations = [c.duration for c in cuts]
plt.hist(durations, bins=8)
plt.xlabel("duration (s)")
plt.ylabel("# clips")
plt.title(f"{len(cuts)} clips, {sum(durations):.0f}s total")
plt.show()

## 3. Prepare training data

Parakeet expects 16kHz mono; our recordings are 32kHz. Lhotse resamples and writes real files for us, then we split into train/eval and export the manifest format NeMo's training loop reads.

In [ ]:
# Split BEFORE resampling/saving: save_audios() below renames each recording_id to match
# cut.id, so filtering by the original filename has to happen first, while recording_id
# still matches EVAL_FILENAME.
eval_cuts_32k = cuts.filter(lambda c: c.recording_id == EVAL_FILENAME).to_eager()
train_cuts_32k = cuts.filter(lambda c: c.recording_id != EVAL_FILENAME).to_eager()

eval_cuts = eval_cuts_32k.resample(16000).save_audios("audio_16k")
train_cuts = train_cuts_32k.resample(16000).save_audios("audio_16k")
len(train_cuts), len(eval_cuts)

In [ ]:
import json
import unicodedata

def strip_diacritics(s):
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

# Sanity check: Parakeet's BPE tokenizer was trained on English and may not represent Czech
# diacritics at all, in which case the training targets degenerate to <unk>/empty and the
# model "learns" to output garbage. Round-trip a sample through the tokenizer to check:
sample_ids = asr_model.tokenizer.text_to_ids(eval_row.text)
print("Tokenizer round-trip check:")
print("  original :", eval_row.text)
print("  roundtrip:", asr_model.tokenizer.ids_to_text(sample_ids))
print("  -> if roundtrip is empty/garbled, that's why training collapses -- we strip")
print("     diacritics below so the training *targets* are at least representable.")

def cuts_to_nemo_manifest(cutset, path):
    with open(path, "w", encoding="utf-8") as f:
        for cut in cutset:
            entry = {
                "audio_filepath": cut.recording.sources[0].source,
                "duration": cut.duration,
                "text": strip_diacritics(cut.supervisions[0].text),
            }
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

cuts_to_nemo_manifest(train_cuts, "train_manifest.json")
cuts_to_nemo_manifest(eval_cuts, "val_manifest.json")
!cat train_manifest.json

## 4. Weights & Biases — watch training happen, not scroll past it

In [ ]:
import wandb

wandb.login()  # pastes an API key from wandb.ai/authorize
wandb.init(project="parakeet-atc-workshop", name="finetune-run")
%wandb

## 5. Finetune

Deliberately tiny: a handful of epochs over ~11 clips, just enough to see the loss move and the outputs shift. Watch the panel above update live while this runs.

In [ ]:
import copy
from omegaconf import open_dict, OmegaConf
import lightning.pytorch as pl
from lightning.pytorch.loggers import WandbLogger

cfg = copy.deepcopy(asr_model.cfg)
with open_dict(cfg):
    cfg.train_ds.manifest_filepath = "train_manifest.json"
    cfg.train_ds.batch_size = 2
    cfg.train_ds.is_tarred = False
    cfg.validation_ds.manifest_filepath = "val_manifest.json"
    cfg.validation_ds.batch_size = 1

asr_model.setup_training_data(cfg.train_ds)
asr_model.setup_validation_data(cfg.validation_ds)

# Configure the optimizer directly, with no "sched" key at all: Lhotse's dynamic dataset has
# no __len__, which NeMo's default scheduler setup needs to compute steps-per-epoch. Popping
# "sched" from asr_model.cfg.optim didn't stick (NeMo reads the scheduler config from
# elsewhere), so instead we hand setup_optimization a fresh, scheduler-free config directly.
# We don't need a schedule for a 15-epoch demo run on 11 clips anyway -- constant LR is enough.
asr_model.setup_optimization(OmegaConf.create({
    "name": "adamw",
    "lr": 1e-5,  # small LR: gently nudge a big pretrained model, don't wreck it
    "betas": [0.9, 0.98],
    "weight_decay": 0.001,
}))

trainer = pl.Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=15,
    logger=WandbLogger(),
    log_every_n_steps=1,
    enable_checkpointing=False,
)
asr_model.set_trainer(trainer)
trainer.fit(asr_model)

## 6. The payoff — before vs. after, same clip

In [ ]:
eval_path = next(iter(eval_cuts)).recording.sources[0].source
finetuned_text = asr_model.transcribe([eval_path])[0].text

print("Reference :", eval_row.text)
print("Baseline  :", baseline_text)
print("Finetuned :", finetuned_text)

In [ ]:
import difflib

def highlight(ref, hyp):
    ref_words, hyp_words = ref.split(), hyp.split()
    sm = difflib.SequenceMatcher(None, ref_words, hyp_words)
    out = []
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        seg = " ".join(hyp_words[j1:j2])
        if tag != "equal" and seg:
            seg = f"<span style='background:#ffd6d6'>{seg}</span>"
        if seg:
            out.append(seg)
    return " ".join(out)

display(HTML(
    f"<p><b>Reference:</b> {eval_row.text}</p>"
    f"<p><b>Baseline (before):</b> {highlight(eval_row.text, baseline_text)}</p>"
    f"<p><b>Finetuned (after):</b> {highlight(eval_row.text, finetuned_text)}</p>"
    "<p><i>Red = doesn't match the reference word-for-word (rough but visual)</i></p>"
))

## Wrap-up — discussion prompts

- What did 12 clips actually buy us? Which words/phrases got picked up, which didn't?
- What would you need to do differently to finetune this properly (data volume, tokenizer/vocab, language)?
- Check the W&B run — did the loss curve even fully converge in 15 epochs on this little data?